# 06 · Green Vehicle Submodel (E85 & CNG Router)
**Project:** Eco-Urbanomics — Vehicle CO2 Emissions

**Why this notebook exists — the failure mode:**  
The main model (notebooks 4/5) has a -49 g/km systematic bias on vehicles below 150 g/km.  
The root cause is a thermodynamic regime mismatch:

| | E85 Ethanol | Regular Gasoline |
|---|---|---|
| CO2 per litre | ~16 g/km per L/100km | ~23 g/km per L/100km |
| Fuel consumption | ~40% higher by volume | baseline |
| Net: what model sees | Low `Fuel_Efficiency_Score` → predicts high CO2 | correct |
| Net: reality | Lower CO2 despite higher consumption | baseline |

The main model was trained mostly on gasoline vehicles (95%) and learned  
"low efficiency score = high emissions." E85 breaks this rule entirely.

**Architecture — two-stage router:**  
1. `Fuel_E` and `Fuel_N` columns (already in processed data) route the vehicle.  
2. Alt-fuel vehicles → E85 submodel (RF + small MLP, trained on 370 E85 rows).  
3. CNG vehicle (n=1) → rule-based override (flag only, no learned model).  
4. All others → existing main model unchanged.  
5. Outputs merged into a single unified predictions CSV.


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, roc_auc_score, mean_squared_error, mean_absolute_error, r2_score
)
import matplotlib.pyplot as plt
import joblib
from pathlib import Path

PROCESSED = Path('../data/processed/processed_co2_data.csv')
MODELS    = Path('../models')
OUTPUTS   = Path('../data/outputs')
SEED      = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

df = pd.read_csv(PROCESSED)
print(f"Full dataset: {df.shape}")

TARGET_C  = 'High_Emitter'
TARGET_R  = 'CO2 Emissions(g/km)'
FEAT_COLS = [c for c in df.columns if c not in [TARGET_C, TARGET_R]]
print(f"Features: {len(FEAT_COLS)}")


In [ ]:
# ── Step 1: Route by fuel type ─────────────────────────────────────────
# Fuel_E = 1 means E85 ethanol; Fuel_N = 1 means CNG natural gas
# Both columns are already in processed_co2_data.csv from notebook 2

mask_e85 = df['Fuel_E'] == 1
mask_cng = df['Fuel_N'] == 1
mask_alt = mask_e85 | mask_cng
mask_main= ~mask_alt

df_e85  = df[mask_e85].copy().reset_index(drop=True)
df_cng  = df[mask_cng].copy().reset_index(drop=True)
df_main = df[mask_main].copy().reset_index(drop=True)

print(f"Route summary:")
print(f"  E85  (submodel) : {len(df_e85):>5} rows  ({len(df_e85)/len(df)*100:.1f}%)")
print(f"  CNG  (override) : {len(df_cng):>5} rows  ({len(df_cng)/len(df)*100:.1f}%)")
print(f"  Main (unchanged): {len(df_main):>5} rows  ({len(df_main)/len(df)*100:.1f}%)")
print()

# Class balance within E85 subset — critical for pos_weight
e85_counts = df_e85[TARGET_C].value_counts()
print(f"E85 class balance:")
print(f"  Normal (0)    : {e85_counts.get(0,0):>4} ({e85_counts.get(0,0)/len(df_e85)*100:.1f}%)")
print(f"  High Emitter  : {e85_counts.get(1,0):>4} ({e85_counts.get(1,0)/len(df_e85)*100:.1f}%)")
print(f"  pos_weight    : {e85_counts.get(0,0)/max(e85_counts.get(1,1),1):.3f}")


In [ ]:
# ── Step 2: Engineer E85-specific features ────────────────────────────
# The key insight: E85 vehicles need a feature that captures
# CO2 output relative to fuel energy content, not just fuel volume.
#
# Ethanol has ~34% less energy per litre than gasoline (21.2 MJ/L vs 32 MJ/L).
# So 16L/100km of E85 delivers the same energy as ~10.5L/100km of gasoline.
# This energy-normalised efficiency is what actually drives CO2 output.

ETHANOL_ENERGY_FACTOR = 0.659  # ethanol energy content / gasoline energy content

def add_e85_features(df_subset):
    df_out = df_subset.copy()

    # Feature A: Energy-adjusted fuel consumption
    # = FC_comb × ethanol_energy_factor (normalised to gasoline-equivalent litres)
    # This is the feature the main model was missing for E85 vehicles
    df_out['FC_energy_adjusted'] = (
        df_out['Fuel Consumption Comb (L/100 km)'] * ETHANOL_ENERGY_FACTOR
    )

    # Feature B: CO2 per energy-adjusted litre
    # Directly captures the thermodynamic regime of E85
    df_out['CO2_per_energy_litre'] = (
        df_out['CO2 Emissions(g/km)'] /
        (df_out['FC_energy_adjusted'] + 1e-6)
    )

    # Feature C: Energy-adjusted efficiency score
    # Replaces the misleading Fuel_Efficiency_Score for E85 vehicles
    df_out['Energy_adj_efficiency'] = 1.0 / (df_out['FC_energy_adjusted'] + 1e-6)

    return df_out

# Apply to E85 subset only
df_e85 = add_e85_features(df_e85)

new_feats = ['FC_energy_adjusted', 'CO2_per_energy_litre', 'Energy_adj_efficiency']
print("E85-specific engineered features:")
print(df_e85[new_feats].describe().T[['mean','std','min','max']].round(4))
print()

# Verify these features separate classes better than original Fuel_Efficiency_Score
for feat in ['Fuel_Efficiency_Score', 'Energy_adj_efficiency']:
    if feat not in df_e85.columns:
        continue
    normal = df_e85[df_e85[TARGET_C]==0][feat].mean()
    high   = df_e85[df_e85[TARGET_C]==1][feat].mean()
    sep    = abs(normal - high) / df_e85[feat].std()
    print(f"  {feat:<30} class sep (Cohen d): {sep:.3f}")


In [ ]:
# ── Step 3: Build E85 feature matrix ──────────────────────────────────
# Use all original features + the 3 new E85-specific ones
# Drop CO2_per_energy_litre from features (it's derived from the target)
E85_FEAT_COLS = FEAT_COLS + ['FC_energy_adjusted', 'Energy_adj_efficiency']

X_e85  = df_e85[E85_FEAT_COLS].values.astype(np.float32)
yc_e85 = df_e85[TARGET_C].values.astype(np.float32)
yr_e85 = df_e85[TARGET_R].values.astype(np.float32)

yr_e85_min, yr_e85_max = yr_e85.min(), yr_e85.max()
yr_e85_norm = (yr_e85 - yr_e85_min) / (yr_e85_max - yr_e85_min)

# Stratified split on the E85 subset
X_tr, X_te, yc_tr, yc_te, yr_tr, yr_te, yrn_tr, yrn_te = train_test_split(
    X_e85, yc_e85, yr_e85, yr_e85_norm,
    test_size=0.20, random_state=SEED, stratify=yc_e85
)
X_tr, X_val, yc_tr, yc_val, yr_tr, yr_val, yrn_tr, yrn_val = train_test_split(
    X_tr, yc_tr, yr_tr, yrn_tr,
    test_size=0.15, random_state=SEED, stratify=yc_tr
)

print(f"E85 splits — Train: {X_tr.shape} | Val: {X_val.shape} | Test: {X_te.shape}")
print(f"E85 CO2 range: {yr_e85_min:.0f} – {yr_e85_max:.0f} g/km")


In [ ]:
# ── Step 4a: Random Forest submodel (E85 baseline) ────────────────────
e85_pos_w = (yc_tr == 0).sum() / max((yc_tr == 1).sum(), 1)
print(f"E85 pos_weight: {e85_pos_w:.3f}")

rf_clf_e85 = RandomForestClassifier(
    n_estimators=300, min_samples_leaf=2,
    class_weight='balanced', random_state=SEED, n_jobs=-1
)
rf_reg_e85 = RandomForestRegressor(
    n_estimators=300, min_samples_leaf=2,
    random_state=SEED, n_jobs=-1
)

rf_clf_e85.fit(X_tr, yc_tr)
rf_reg_e85.fit(X_tr, yr_tr)

# Evaluate RF submodel
yc_pred_rf = rf_clf_e85.predict(X_te)
yc_prob_rf = rf_clf_e85.predict_proba(X_te)[:, 1]
yr_pred_rf = rf_reg_e85.predict(X_te)

print("\n" + "="*50)
print("  E85 RF SUBMODEL — Classification")
print("="*50)
print(classification_report(yc_te, yc_pred_rf,
      target_names=['Normal','High Emitter']))
print(f"  Macro F1 : {f1_score(yc_te, yc_pred_rf, average='macro'):.4f}")
print(f"  ROC-AUC  : {roc_auc_score(yc_te, yc_prob_rf):.4f}")

rmse_rf = np.sqrt(mean_squared_error(yr_te, yr_pred_rf))
r2_rf   = r2_score(yr_te, yr_pred_rf)
print(f"\n  Regression — RMSE: {rmse_rf:.2f} g/km  R²: {r2_rf:.4f}")


In [ ]:
# ── Step 4b: Small MLP submodel (E85) ─────────────────────────────────
class E85Dataset(Dataset):
    def __init__(self, X, yc, yr_norm):
        self.X  = torch.from_numpy(X)
        self.yc = torch.from_numpy(yc).unsqueeze(1)
        self.yr = torch.from_numpy(yr_norm).unsqueeze(1)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.yc[i], self.yr[i]

# Smaller network than main model — E85 subset is only ~250 training rows
class E85MLP(nn.Module):
    """
    Compact dual-head MLP for E85 vehicles.
    Smaller than the main model (128→64→32) to avoid overfitting on ~250 rows.
    Higher dropout (0.4) for the same reason.
    """
    def __init__(self, in_dim, dropout=0.4):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(in_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64),     nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64,  32),     nn.BatchNorm1d(32),  nn.ReLU(), nn.Dropout(dropout),
        )
        self.clf_head = nn.Sequential(nn.Linear(32, 1))
        self.reg_head = nn.Sequential(nn.Linear(32, 1), nn.Sigmoid())

    def forward(self, x):
        s = self.trunk(x)
        return self.clf_head(s), self.reg_head(s)

BATCH = 16   # small batch for small dataset
train_loader = DataLoader(E85Dataset(X_tr,  yc_tr,  yrn_tr),  batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(E85Dataset(X_val, yc_val, yrn_val), batch_size=BATCH, shuffle=False)
test_loader  = DataLoader(E85Dataset(X_te,  yc_te,  yrn_te),  batch_size=BATCH, shuffle=False)

model_e85 = E85MLP(in_dim=X_tr.shape[1]).to(DEVICE)
pos_w_t   = torch.tensor([e85_pos_w], dtype=torch.float32).to(DEVICE)
clf_crit  = nn.BCEWithLogitsLoss(pos_weight=pos_w_t)
reg_crit  = nn.MSELoss()
optimizer = optim.AdamW(model_e85.parameters(), lr=5e-4, weight_decay=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=15, verbose=False
)

print(f"E85 MLP params: {sum(p.numel() for p in model_e85.parameters() if p.requires_grad):,}")
print(f"Batch size: {BATCH} (small — dataset is only {len(X_tr)} training rows)")


In [ ]:
# ── Train E85 MLP ─────────────────────────────────────────────────────
def run_epoch_e85(loader, train=True):
    model_e85.train(train)
    total_loss, all_preds, all_probs, all_true = 0.0, [], [], []
    all_reg_pred, all_reg_true = [], []

    with torch.set_grad_enabled(train):
        for xb, yc_b, yr_b in loader:
            xb, yc_b, yr_b = xb.to(DEVICE), yc_b.to(DEVICE), yr_b.to(DEVICE)
            clf_logit, reg_out = model_e85(xb)
            loss = 0.7 * clf_crit(clf_logit, yc_b) + 0.3 * reg_crit(reg_out, yr_b)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * len(xb)
            probs = torch.sigmoid(clf_logit).cpu().detach().numpy().flatten()
            all_probs.extend(probs)
            all_preds.extend((probs >= 0.5).astype(float))
            all_true.extend(yc_b.cpu().numpy().flatten())
            all_reg_pred.extend(reg_out.cpu().detach().numpy().flatten())
            all_reg_true.extend(yr_b.cpu().numpy().flatten())

    return (total_loss / len(loader.dataset),
            f1_score(all_true, all_preds, average='macro', zero_division=0),
            np.sqrt(mean_squared_error(all_reg_true, all_reg_pred)))

EPOCHS, PATIENCE = 300, 40   # more epochs — small dataset converges slowly
best_f1, best_ep, no_imp = 0.0, 0, 0
history = {k: [] for k in ['tr_loss','vl_loss','tr_f1','vl_f1']}

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_f1, _ = run_epoch_e85(train_loader, train=True)
    vl_loss, vl_f1, _ = run_epoch_e85(val_loader,   train=False)
    for k, v in zip(['tr_loss','vl_loss','tr_f1','vl_f1'],
                    [tr_loss, vl_loss, tr_f1, vl_f1]):
        history[k].append(v)
    scheduler.step(vl_f1)
    if vl_f1 > best_f1:
        best_f1, best_ep, no_imp = vl_f1, epoch, 0
        torch.save(model_e85.state_dict(), MODELS / 'e85_submodel_nn.pth')
    else:
        no_imp += 1
    if epoch % 50 == 0 or epoch == 1:
        print(f"Ep {epoch:3d} | Train F1={tr_f1:.4f} loss={tr_loss:.4f} | "
              f"Val F1={vl_f1:.4f} loss={vl_loss:.4f} | LR={optimizer.param_groups[0]['lr']:.2e}")
    if no_imp >= PATIENCE:
        print(f"\nEarly stop @ epoch {epoch}. Best val F1={best_f1:.4f} @ epoch {best_ep}")
        break

print(f"\nBest checkpoint: epoch {best_ep}, val F1 = {best_f1:.4f}")


In [ ]:
# ── Evaluate E85 MLP ──────────────────────────────────────────────────
model_e85.load_state_dict(torch.load(MODELS / 'e85_submodel_nn.pth', map_location=DEVICE))
model_e85.eval()

mlp_probs, mlp_preds, mlp_true = [], [], []
mlp_reg_pred, mlp_reg_true     = [], []

with torch.no_grad():
    for xb, yc_b, yr_b in test_loader:
        xb = xb.to(DEVICE)
        clf_logit, reg_out = model_e85(xb)
        probs = torch.sigmoid(clf_logit).cpu().numpy().flatten()
        mlp_probs.extend(probs)
        mlp_preds.extend((probs >= 0.5).astype(float))
        mlp_true.extend(yc_b.numpy().flatten())
        mlp_reg_pred.extend(reg_out.cpu().numpy().flatten())
        mlp_reg_true.extend(yr_b.numpy().flatten())

# Denormalise regression
mlp_co2_pred = np.array(mlp_reg_pred) * (yr_e85_max - yr_e85_min) + yr_e85_min
mlp_co2_true = np.array(mlp_reg_true) * (yr_e85_max - yr_e85_min) + yr_e85_min

print("="*50)
print("  E85 MLP SUBMODEL — Classification")
print("="*50)
print(classification_report(mlp_true, mlp_preds,
      target_names=['Normal','High Emitter']))
print(f"  Macro F1 : {f1_score(mlp_true, mlp_preds, average='macro'):.4f}")
print(f"  ROC-AUC  : {roc_auc_score(mlp_true, mlp_probs):.4f}")
rmse_mlp = np.sqrt(mean_squared_error(mlp_co2_true, mlp_co2_pred))
r2_mlp   = r2_score(mlp_co2_true, mlp_co2_pred)
print(f"\n  Regression — RMSE: {rmse_mlp:.2f} g/km  R²: {r2_mlp:.4f}")


In [ ]:
# ── Step 5: CNG rule-based override ───────────────────────────────────
# CNG (n=1 in dataset — one Chevrolet Bi-Fuel) cannot be learned from.
# Rule: flag any CNG vehicle with a warning and use FC_comb as a proxy.
# CNG emits ~15.3 g/km per L/100km equivalent (vs 23.3 for gasoline).
CNG_CO2_PER_FC = 15.3   # g/km per L/100km, from EPA CNG emission factors

def predict_cng(df_cng_subset):
    """Rule-based CNG predictor using emission factor from EPA data."""
    if len(df_cng_subset) == 0:
        return pd.DataFrame()
    preds = []
    for _, row in df_cng_subset.iterrows():
        fc_comb   = row.get('Fuel Consumption Comb (L/100 km)', 12.7)
        co2_est   = fc_comb * CNG_CO2_PER_FC
        high_flag = int(co2_est >= 278)
        preds.append({
            'source'              : 'CNG_rule_based',
            'Actual_CO2_gkm'      : row.get('CO2 Emissions(g/km)', np.nan),
            'Predicted_CO2_q50'   : round(co2_est, 1),
            'Predicted_High_Emitter': high_flag,
            'Emission_Risk_Score' : float(high_flag),
            'note'                : 'CNG rule-based — insufficient data for learned model'
        })
    return pd.DataFrame(preds)

cng_preds = predict_cng(df_cng)
print("CNG predictions (rule-based):")
print(cng_preds.to_string(index=False))


In [ ]:
# ── Step 6: Evaluation plots ───────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(17, 10))

# 1. Training curves
ax = axes[0, 0]
ax.plot(history['tr_f1'], color='#E89A4C', label='Train F1')
ax.plot(history['vl_f1'], color='#4C8BE8', alpha=0.85, label='Val F1')
ax.axvline(best_ep - 1, color='red', linestyle='--', linewidth=1.2,
           label=f'Best ep {best_ep}')
ax.set_title('E85 MLP — training curve', fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Macro F1')
ax.legend(fontsize=9); ax.spines[['top','right']].set_visible(False)

# 2. Confusion matrix — MLP submodel
ConfusionMatrixDisplay(
    confusion_matrix(mlp_true, mlp_preds),
    display_labels=['Normal','High Emitter']
).plot(ax=axes[0, 1], cmap='Blues', colorbar=False)
axes[0, 1].set_title('E85 MLP — confusion matrix', fontweight='bold')

# 3. Actual vs Predicted CO2 — MLP submodel
ax = axes[0, 2]
ax.scatter(mlp_co2_true, mlp_co2_pred, alpha=0.5, s=20, color='#4C8BE8')
lims = [min(mlp_co2_true.min(), mlp_co2_pred.min()),
        max(mlp_co2_true.max(), mlp_co2_pred.max())]
ax.plot(lims, lims, 'r--', linewidth=1.5)
ax.set_xlabel('Actual CO2 (g/km)'); ax.set_ylabel('Predicted CO2 (g/km)')
ax.set_title(f'E85 MLP regression — R²={r2_mlp:.4f}', fontweight='bold')
ax.spines[['top','right']].set_visible(False)

# 4. Bias comparison: main model vs E85 submodel on the same E85 test vehicles
# Load main model predictions on E85 vehicles for comparison
main_model_path = MODELS / 'carbon_predictor_nn.pth'
if main_model_path.exists():
    # Re-run main model on E85 test features (without energy features)
    class MainMLP(nn.Module):
        def __init__(self, in_dim, hidden=(256,128,64), dropout=0.3):
            super().__init__()
            trunk, prev = [], in_dim
            for h in hidden:
                trunk += [nn.Linear(prev,h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
                prev = h
            self.trunk    = nn.Sequential(*trunk)
            self.clf_head = nn.Sequential(nn.Linear(prev,32), nn.ReLU(), nn.Dropout(0.2), nn.Linear(32,1))
            self.reg_head = nn.Sequential(nn.Linear(prev,32), nn.ReLU(), nn.Dropout(0.2), nn.Linear(32,1), nn.Sigmoid())
        def forward(self, x): s=self.trunk(x); return self.clf_head(s), self.reg_head(s)

    # Load main model's normalisation params from its training range
    yr_main     = df_main[TARGET_R].values.astype(np.float32)
    yr_main_min, yr_main_max = yr_main.min(), yr_main.max()

    main_model = MainMLP(in_dim=len(FEAT_COLS)).to(DEVICE)
    main_model.load_state_dict(torch.load(main_model_path, map_location=DEVICE))
    main_model.eval()

    X_e85_main = torch.from_numpy(
        df_e85.iloc[df_e85.index.isin(
            # reconstruct test indices via same seed
            train_test_split(range(len(df_e85)), test_size=0.2, random_state=SEED)[1]
        )][FEAT_COLS].values.astype(np.float32)
    ).to(DEVICE)

    with torch.no_grad():
        _, reg_main = main_model(X_e85_main)
    main_co2 = reg_main.cpu().numpy().flatten() * (yr_main_max - yr_main_min) + yr_main_min

    residuals_main = mlp_co2_true[:len(main_co2)] - main_co2
    residuals_sub  = mlp_co2_true - mlp_co2_pred

    ax = axes[1, 0]
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.scatter(mlp_co2_true[:len(main_co2)], residuals_main,
               alpha=0.5, s=15, color='#E84C4C', label=f'Main model (bias={residuals_main.mean():+.1f})')
    ax.scatter(mlp_co2_true, residuals_sub,
               alpha=0.5, s=15, color='#4C8BE8', label=f'E85 submodel (bias={residuals_sub.mean():+.1f})')
    ax.set_xlabel('Actual CO2 (g/km)')
    ax.set_ylabel('Residual (actual − predicted, g/km)')
    ax.set_title('Residual comparison on E85 test set', fontweight='bold')
    ax.legend(fontsize=9); ax.spines[['top','right']].set_visible(False)
else:
    axes[1, 0].text(0.5, 0.5, 'Run notebook 4 first\nto generate carbon_predictor_nn.pth',
                   ha='center', va='center', transform=axes[1,0].transAxes, fontsize=11)
    axes[1, 0].set_title('Residual comparison (needs nb4)', fontweight='bold')

# 5. Feature importances (RF submodel)
ax = axes[1, 1]
feat_imp = pd.Series(rf_clf_e85.feature_importances_, index=E85_FEAT_COLS).sort_values(ascending=True)
colors = ['#E84C4C' if any(k in n for k in ['energy','Energy','Green','Fuel_E']) else '#4C8BE8'
          for n in feat_imp.index]
feat_imp.tail(15).plot(kind='barh', ax=ax, color=colors[-15:], edgecolor='white')
ax.set_title('E85 RF — feature importances (top 15)', fontweight='bold')
ax.set_xlabel('Mean Decrease in Impurity')
ax.spines[['top','right']].set_visible(False)

# 6. RF submodel regression
ax = axes[1, 2]
yr_pred_rf_denorm = yr_pred_rf * (yr_e85_max - yr_e85_min) + yr_e85_min
yr_te_denorm      = yr_te      * (yr_e85_max - yr_e85_min) + yr_e85_min
ax.scatter(yr_te_denorm, yr_pred_rf_denorm, alpha=0.5, s=20, color='#E89A4C')
lims2 = [min(yr_te_denorm.min(), yr_pred_rf_denorm.min()),
         max(yr_te_denorm.max(), yr_pred_rf_denorm.max())]
ax.plot(lims2, lims2, 'r--', linewidth=1.5)
ax.set_xlabel('Actual CO2 (g/km)'); ax.set_ylabel('Predicted CO2 (g/km)')
ax.set_title(f'E85 RF regression — R²={r2_rf:.4f}', fontweight='bold')
ax.spines[['top','right']].set_visible(False)

plt.suptitle('Notebook 6: Green Vehicle Submodel Results', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUTS / 'green_submodel_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Step 7: Unified router — inference function ────────────────────────
def unified_predict(df_input: pd.DataFrame) -> pd.DataFrame:
    """
    Route each vehicle to the correct model and return a unified predictions DataFrame.

    Usage:
        new_vehicles = pd.read_csv('../data/processed/processed_co2_data.csv')
        results = unified_predict(new_vehicles)

    Returns columns:
        source                  — which model handled this vehicle
        Actual_CO2_gkm          — true value (if available)
        Predicted_CO2_q50       — point estimate (g/km)
        Predicted_High_Emitter  — 0 or 1
        Emission_Risk_Score     — probability [0, 1]
    """
    results = []

    for _, row in df_input.iterrows():
        row_feats = row[FEAT_COLS].values.astype(np.float32)

        # ── Route ────────────────────────────────────────────────────
        is_cng = row.get('Fuel_N', 0) == 1
        is_e85 = row.get('Fuel_E', 0) == 1

        if is_cng:
            # Rule-based CNG override
            fc   = row.get('Fuel Consumption Comb (L/100 km)', 12.7)
            co2  = fc * CNG_CO2_PER_FC
            pred = {'source': 'CNG_rule',
                    'Predicted_CO2_q50': round(co2, 1),
                    'Predicted_High_Emitter': int(co2 >= 278),
                    'Emission_Risk_Score': float(int(co2 >= 278))}

        elif is_e85:
            # E85 submodel (MLP)
            e85_row   = row.copy()
            fc_comb   = row.get('Fuel Consumption Comb (L/100 km)', 16.9)
            e85_row['FC_energy_adjusted']   = fc_comb * ETHANOL_ENERGY_FACTOR
            e85_row['Energy_adj_efficiency'] = 1.0 / (fc_comb * ETHANOL_ENERGY_FACTOR + 1e-6)

            x_t = torch.tensor(
                e85_row[E85_FEAT_COLS].values.astype(np.float32)
            ).unsqueeze(0).to(DEVICE)

            model_e85.eval()
            with torch.no_grad():
                clf_logit, reg_out = model_e85(x_t)
            prob  = torch.sigmoid(clf_logit).item()
            co2   = reg_out.item() * (yr_e85_max - yr_e85_min) + yr_e85_min
            pred  = {'source': 'E85_submodel',
                     'Predicted_CO2_q50': round(co2, 1),
                     'Predicted_High_Emitter': int(prob >= 0.5),
                     'Emission_Risk_Score': round(prob, 4)}
        else:
            # Main model (notebook 4)
            x_t = torch.tensor(row_feats).unsqueeze(0).to(DEVICE)
            main_model.eval()
            with torch.no_grad():
                clf_logit, reg_out = main_model(x_t)
            prob  = torch.sigmoid(clf_logit).item()
            co2   = reg_out.item() * (yr_main_max - yr_main_min) + yr_main_min
            pred  = {'source': 'main_model',
                     'Predicted_CO2_q50': round(co2, 1),
                     'Predicted_High_Emitter': int(prob >= 0.5),
                     'Emission_Risk_Score': round(prob, 4)}

        pred['Actual_CO2_gkm'] = row.get(TARGET_R, np.nan)
        results.append(pred)

    return pd.DataFrame(results)


# ── Test on a sample of 30 vehicles covering all three routes ──────────
sample = pd.concat([
    df_e85.sample(10, random_state=SEED),
    df_cng,
    df_main.sample(19, random_state=SEED)
]).reset_index(drop=True)

results = unified_predict(sample)
print("Unified router test (30 vehicles):")
print(results.groupby('source')[['Predicted_CO2_q50','Emission_Risk_Score']].describe().round(1))
print()
print("Route distribution:")
print(results['source'].value_counts())


In [ ]:
# ── Step 8: Save models + full unified predictions ────────────────────
joblib.dump(rf_clf_e85, MODELS / 'e85_submodel_rf_clf.pkl')
joblib.dump(rf_reg_e85, MODELS / 'e85_submodel_rf_reg.pkl')
print(f"Saved RF submodels → {MODELS}")
print(f"Saved MLP submodel → {MODELS / 'e85_submodel_nn.pth'}")

# Generate unified predictions for the full dataset
print("\nGenerating unified predictions for full dataset...")
all_preds = unified_predict(df)
all_preds.to_csv(OUTPUTS / 'unified_predictions_output.csv', index=False)
print(f"Saved → {OUTPUTS / 'unified_predictions_output.csv'}")
print(f"Shape : {all_preds.shape}")
print()
print("Route breakdown:")
print(all_preds['source'].value_counts())
print()

# Final bias check: main model vs submodel on E85 vehicles
e85_results = all_preds[all_preds['source'] == 'E85_submodel'].copy()
if 'Actual_CO2_gkm' in e85_results.columns:
    e85_results = e85_results.dropna(subset=['Actual_CO2_gkm'])
    bias = (e85_results['Predicted_CO2_q50'] - e85_results['Actual_CO2_gkm']).mean()
    rmse = np.sqrt(((e85_results['Predicted_CO2_q50'] - e85_results['Actual_CO2_gkm'])**2).mean())
    print(f"E85 submodel final bias : {bias:+.2f} g/km  (was ~−49 g/km with main model)")
    print(f"E85 submodel final RMSE : {rmse:.2f} g/km")


## Summary

| Vehicle type | Route | Model | Training data |
|---|---|---|---|
| E85 ethanol (n=370) | `Fuel_E == 1` | E85 RF + MLP submodel | 370 E85 rows |
| CNG natural gas (n=1) | `Fuel_N == 1` | Rule-based override | EPA emission factor |
| Gasoline / Diesel (n=7,014) | default | Main model (nb 4/5) | 7,014 rows |

**New engineered features for E85 only:**
- `FC_energy_adjusted` = FC_comb × 0.659 (corrects for ethanol's lower energy density)
- `Energy_adj_efficiency` = 1 / FC_energy_adjusted (replaces misleading `Fuel_Efficiency_Score`)

**Files added to `models/`:**
- `e85_submodel_nn.pth` — MLP weights
- `e85_submodel_rf_clf.pkl` — RF classifier
- `e85_submodel_rf_reg.pkl` — RF regressor

**Files added to `data/outputs/`:**
- `green_submodel_evaluation.png`
- `unified_predictions_output.csv` — full dataset predictions with `source` column

**Expected improvement:** E85 bias drops from −49 g/km (main model) to near 0 g/km.  
RMSE on E85 vehicles should fall from ~55 g/km to ~15 g/km.
